# San Francisco Assessor Roll Data Analysis

This notebook analyzes the San Francisco historical secured property tax roll data for visualization and modeling. The main goal is to turn a large administrative dataset into a clean, interpretable geospatial property dataset that can later support visualization, neighborhood comparison, and price-related modeling.

## What this notebook does
1. Loads the raw assessor roll and applies an explicit schema so columns keep the intended data types.
2. Parses the geometry column and converts the table into a geospatially usable structure.
3. Checks whether small assessment components such as fixtures and personal property are material enough to include.
4. Builds a working measure of total assessed value.
5. Summarizes the composition of San Francisco's property stock by broad property type.
6. Examines how property counts and assessed values evolve over time.
7. Focuses on single-family homes and condominiums, since these are the most relevant categories for later housing-focused work.
8. Compares neighborhoods using assessed value, assessed value per square foot, building age, and recent transaction activity.

## Why this matters
Raw assessor data is rich but not immediately analysis-ready. Important fields require cleaning, category consolidation, geospatial parsing, and careful interpretation. This notebook establishes a transparent foundation so that later modeling work is based on clearly documented assumptions rather than opaque preprocessing steps.

## Important interpretation note
This dataset contains **assessed values**, not a direct market sale-price series. In California, Proposition 13 strongly affects how assessed values evolve over time. That means the notebook treats assessed value as a useful proxy for long-run property value, while also separately examining properties with more recent transactions to get closer to current market conditions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import geopandas as gpd
from shapely import wkt
import contextily as cx
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from IPython.display import display


In [ ]:
# ---------------- #
# PARAMETER SET UP #
# ---------------- #

# You should download the data from: https://data.sfgov.org/d/wv5m-vpq2
# The full dataset is over 1G in size.
INPUT_DATA_CSV = 'raw_data/wv5m-vpq2_version_10715.csv'

In [ ]:
# ------------------- #
# style and format    #
# ------------------- #

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 130,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.title_fontsize": 10,
})

def fmt_int(value):
    return f"{int(value):,}" if pd.notna(value) else "—"

def fmt_pct(value, digits=1):
    return f"{value:.{digits}f}%" if pd.notna(value) else "—"

def fmt_usd(value, digits=0):
    return f"${value:,.{digits}f}" if pd.notna(value) else "—"

def fmt_usd_m(value, digits=3):
    return f"${value:,.{digits}f}M" if pd.notna(value) else "—"

def fmt_usd_b(value, digits=3):
    return f"${value:,.{digits}f}B" if pd.notna(value) else "—"

def fmt_psf(value, digits=0):
    return f"${value:,.{digits}f} / sq ft" if pd.notna(value) else "—"

def fmt_years(value, digits=0):
    return f"{value:,.{digits}f} years" if pd.notna(value) else "—"

def style_table(df, caption=None, formatters=None):
    styler = (
        df.style
        .format(formatters or {})
        .set_properties(**{
            "text-align": "left",
            "font-size": "11pt",
            "padding": "6px 8px",
            "white-space": "nowrap",
        })
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "left"), ("font-size", "11pt"), ("padding", "6px 8px")]},
            {"selector": "caption", "props": [("caption-side", "top"), ("text-align", "left"), ("font-size", "12pt"), ("font-weight", "bold"), ("padding", "0 0 8px 0")]},
        ])
    )
    if caption:
        styler = styler.set_caption(caption)
    return styler

def clean_year_index(index_like):
    return pd.Index(pd.to_datetime(index_like).year.astype(str), name="Roll Year")

def apply_standard_axis_style(ax, y_is_currency=False, currency_scale=1, y_suffix="", rotate_x=45):
    ax.grid(axis="y", alpha=0.2, linewidth=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="x", rotation=rotate_x)
    if y_is_currency:
        if currency_scale == 1_000_000_000:
            ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"${x/1_000_000_000:,.0f}B"))
        elif currency_scale == 1_000_000:
            ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"${x/1_000_000:,.1f}M"))
        else:
            ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"${x:,.0f}{y_suffix}"))
    else:
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:,.0f}{y_suffix}"))


In [ ]:
# when importing, make sure the data types for the fields are set properly
field_dtype = {
    'closed_roll_year': 'Int64',
    'property_location': 'string',
    'parcel_number': 'string',
    'block': 'string',
    'lot': 'string',
    'volume_number': 'Int64',
    'use_code': 'string',
    'use_definition': 'string',
    'property_class_code': 'string',
    'property_class_code_definition': 'string',
    'year_property_built': 'Int64',
    'number_of_bathrooms': 'Int64',
    'number_of_bedrooms': 'Int64',
    'number_of_rooms': 'Int64',
    'number_of_stories': 'Int64',
    'number_of_units': 'Int64',
    'zoning_code': 'string',
    'construction_type': 'string',
    'lot_code': 'string',
    'tax_rate_area_code': 'string',
    'exemption_code': 'string',
    'exemption_code_definition': 'string',
    'status_code': 'string',
    'assessor_neighborhood_district': 'string',
    'assessor_neighborhood_code': 'string',
    'assessor_neighborhood': 'string',
    'supervisor_district': 'string',
    'supervisor_district_2012': 'string',
    'analysis_neighborhood': 'string',
    'row_id': 'string'
 }


df = pd.read_csv(INPUT_DATA_CSV, dtype=field_dtype, parse_dates=["current_sales_date", "data_as_of", "data_loaded_at"])

In [ ]:
# format the year properly
df['closed_roll_year'] = pd.to_datetime(df['closed_roll_year'], format="%Y")

# clean up the geometry
df.dropna(subset=['the_geom'], inplace=True)
df['geometry'] = df['the_geom'].apply(wkt.loads)
df.drop(['the_geom'], axis=1, inplace=True)

In [ ]:
############
# OPTIONAL #
############
# extract and store the definitions of various codes used in the dataset
# so that we can drop certain columns without losing information

# definition_use_code = df[['use_code', 'use_definition']].drop_duplicates().dropna().sort_values('use_code')
# definition_use_code.to_csv('definition/definition_use_code.csv', index=False)

# definition_property_class_code = df[['use_code', 'property_class_code', 'property_class_code_definition']].drop_duplicates().dropna().sort_values(['use_code','property_class_code'])
# definition_property_class_code.to_csv('definition/definition_property_class_code.csv', index=False)

# definition_exemption_code = df[['exemption_code', 'exemption_code_definition']].drop_duplicates().dropna().sort_values('exemption_code')
# definition_exemption_code.to_csv('definition/definition_exemption_code.csv', index=False)

# definition_assessor_neighborhood_code = df[['assessor_neighborhood_district', 'assessor_neighborhood_code', 'assessor_neighborhood']].drop_duplicates().dropna().sort_values('assessor_neighborhood_code')
# definition_assessor_neighborhood_code.to_csv('definition/definition_assessor_neighborhood_code.csv', index=False)

## 0(A). Validate whether fixture and personal-property assessments can be ignored

Before constructing a simplified total value measure, it is useful to check whether two smaller assessed components—**fixtures** and **personal property**—are common or economically meaningful in this dataset.

The purpose of this step is methodological: if these fields are almost always zero or trivial, excluding them makes the downstream analysis cleaner without materially changing conclusions. That helps keep the value measure focused on the two core components of real estate taxation: **land value** and **improvement value**.

In [ ]:
summary = pd.DataFrame({
    "Share of Records with Non-Zero Value": [
        (df["assessed_fixtures_value"] > 0).mean() * 100,
        (df["assessed_personal_property_value"] > 0).mean() * 100,
    ],
    "Median Assessed Amount (USD)": [
        df["assessed_fixtures_value"].median(),
        df["assessed_personal_property_value"].median(),
    ],
}, index=["Fixtures", "Personal property"])

display(
    style_table(
        summary,
        caption="Materiality check for fixture and personal-property assessments",
        formatters={
            "Share of Records with Non-Zero Value": lambda x: fmt_pct(x, 2),
            "Median Assessed Amount (USD)": fmt_usd,
        },
    )
)


#### Observation and decision

The auxiliary assessment fields are present, but they are economically minor at the citywide level.

- Only **0.64%** of records have a non-zero **fixture** assessment, and only **4.68%** have a non-zero **personal-property** assessment.
- The **median assessed amount is $0** for both fields, which means the typical parcel does not carry either component.
- For broad spatial and cross-sectional analysis, excluding these fields introduces very little distortion while keeping the valuation logic transparent.

**Modeling choice used throughout the notebook:** total assessed value is defined as **land value + improvement value**.


## 0(B). Construct total assessed value

This step creates the core value variable used throughout the notebook:

- **`assessed_total`** = land assessment + improvement assessment
- **`log10_assessed_total`** = log10-scaled version used for better distributional behavior when needed

The raw assessed-value distribution is highly skewed, so keeping both the original scale and a log10 version is useful. The original scale is easier to interpret in dollars, while the log10 scale is often better for modeling and visualization when extreme values are present. A pseudo count of 1 is added to avoid applying log10 to number 0.

In [ ]:
df["assessed_total"] = df["assessed_land_value"].fillna(0) + df["assessed_improvement_value"].fillna(0)
df["log10_assessed_total"] = np.log10(df["assessed_total"] + 1)

## Analysis: number and proportion of property types

The assessor roll contains many detailed property classifications. For a citywide structural view, the notebook groups them into broader categories such as single-family residential, multi-family residential, retail, office, hotel, industrial, and public/vacant land.

This section answers two basic but important questions:

1. **What kinds of properties make up San Francisco's built environment?**
2. **How has that composition changed over time?**

### Category notes
- **Single Family Residential (SRES) - '1 Single Family (SRES)':** mainly dwellings, condominiums, and town houses.
- **Multi Family Residential (MRES) - '2 Multi Family (MRES)':** mainly flats, duplexes, apartments, and mixed flat/store buildings.
- **Commercial Retail (COMR) - '3 Retail (COMR)':** mainly commercial stores and store units in condominiums.
- **Commercial Office (COMO) - '4 Office (COMO)':** mainly office buildings, government offices, and office units in condominiums.
- **Commercial Hotel (COMH) - '5 Hotel (COMH)':** mainly hotels and motels.
- **Commercial Misc (COMM) - '6 Timeshare (COMM)':** mainly timeshare properties.
- **Industrial (IND) - '7 Industrial (IND)':** mainly industrial and warehouse properties.
- **Government (GOVT) - '8 Vacant Gov (GOVT)':** mainly government-owned vacant land.
- **Miscellaneous/Mixed-Use (MISC) - '9 Vacant Public (MISC)':** mainly public-use or residential vacant lots.

Collapsing the detailed use codes into a smaller set of analytically meaningful groups makes later charts much easier to read and compare.

In [ ]:
property_type_lookup = (
    df[["use_code", "use_definition"]]
    .drop_duplicates()
    .dropna()
    .sort_values("use_code", ascending=False)
    .rename(columns={
        "use_code": "Use Code",
        "use_definition": "Property Type Description",
    })
    .reset_index(drop=True)
)

display(style_table(property_type_lookup, caption="Property type codes appearing in the assessor roll"))


In [ ]:
# rename to terms easier to understand
use_code_maps = {'SRES':'1 Single Family (SRES)',
                 'MRES':'2 Multi Family (MRES)',
                 'COMR':'3 Retail (COMR)',
                 'COMO':'4 Office (COMO)', 
                 'COMH':'5 Hotel (COMH)', 
                 'COMM':'6 Timeshare (COMM)', 
                 'IND':'7 Industrial (IND)', 
                 'GOVT':'8 Vacant Gov (GOVT)', 
                 'MISC':'9 Vacant Public (MISC)'}

## 1. Where are the various property types located in San Francisco?

A citywide location map helps connect property use to urban structure. This is important because counts alone do not show whether a category is diffuse, corridor-based, waterfront-oriented, or concentrated in the downtown core.

By plotting the 2024 property records spatially, we can compare the residential fabric of western and southern neighborhoods with the more commercial and industrial concentration in central and eastern San Francisco.

In [ ]:
df_2024 = df.loc[df['closed_roll_year']=='2024-01-01'].copy()
df_2024['use_code'] = df_2024['use_code'].map(use_code_maps)

## add a bit more information
df_2024['assessor_neighborhood'] = df_2024['assessor_neighborhood_code'] + ' ' + df_2024['assessor_neighborhood'] 

In [ ]:
# -------------------------------------------------------------------
# Prepare geospatial data
# -------------------------------------------------------------------
gdf_2024 = gpd.GeoDataFrame(
    df_2024[['use_code', 'geometry']],
    geometry='geometry',
    crs='EPSG:4326'
)

data_3857 = gdf_2024.to_crs(epsg=3857)

color_mapping = {
    '1 Single Family (SRES)': 'C0',
    '2 Multi Family (MRES)': 'C1',
    '3 Retail (COMR)': 'C2',
    '4 Office (COMO)': 'C3',
    '5 Hotel (COMH)': 'C4',
    '6 Timeshare (COMM)': 'C5',
    '7 Industrial (IND)': 'C6',
    '8 Vacant Gov (GOVT)': 'C7',
    '9 Vacant Public (MISC)': 'C8'
}

# -------------------------------------------------------------------
# Plot
# -------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 10))

data_3857.plot(
    ax=ax,
    color=data_3857['use_code'].map(color_mapping),
    markersize=2,
    alpha=0.9,
    zorder=3,
)

cx.add_basemap(
    ax,
    source=cx.providers.CartoDB.Positron,
    attribution=False
)

ax.set_title(
    'Location of Various Property Types in San Francisco',
    fontsize=14,
    pad=8
)
ax.set_axis_off()

# -------------------------------------------------------------------
# Tight map extent with reduced outer whitespace
# -------------------------------------------------------------------
xmin, ymin, xmax, ymax = data_3857.total_bounds
xpad = (xmax - xmin) * 0.01
ypad = (ymax - ymin) * 0.01

ax.set_xlim(xmin - xpad, xmax + xpad)
ax.set_ylim(ymin - ypad, ymax + ypad)

# -------------------------------------------------------------------
# Legend inside figure
# -------------------------------------------------------------------
legend_handles = [
    Line2D(
        [0], [0],
        marker='o',
        color='w',
        label=label,
        markerfacecolor=color,
        markersize=8
    )
    for label, color in color_mapping.items()
]

ax.legend(
    handles=legend_handles,
    title='Property Type',
    loc='upper left',
    frameon=True,
    framealpha=0.95,
    facecolor='white',
    edgecolor='0.8',
    fontsize=9,
    title_fontsize=10
)

# -------------------------------------------------------------------
# Layout and export
# -------------------------------------------------------------------
plt.subplots_adjust(left=0.02, right=0.98, top=0.94, bottom=0.02)

# Uncomment to save
plt.savefig(
     'assets/01_citywide_property_type_map.png',
     dpi=300,
     bbox_inches='tight',
     pad_inches=0.02
)

plt.show()

In [ ]:
# For each property type, identify the top 5 neighborhoods by record count in 2024.
top5 = (
    df_2024
    .value_counts(["use_code", "assessor_neighborhood"])
    .groupby(level=0)
    .head(5)
    .reset_index(name="property_count")
)

top5["rank"] = top5.groupby("use_code").cumcount() + 1

top5_pivot = top5.pivot(index="rank", columns="use_code", values="assessor_neighborhood")
top5_pivot.index.name = "Rank"
top5_pivot.columns.name = "Property Type"

display(style_table(top5_pivot, caption="Top 5 neighborhoods by property count within each property type"))


#### Observations

The neighborhood concentration pattern is highly consistent with San Francisco's built form.

- **Single-family homes** are most common in outer and hill-oriented residential neighborhoods such as **Bernal Heights, Parkside, Central Sunset, and Excelsior**.
- **Multi-family housing** is densest in urban neighborhoods such as **Inner Mission, Central Richmond, Noe Valley, Inner Richmond, and Inner Sunset**, which fits the city's corridor-style apartment geography.
- **Office and hotel uses** cluster in the historic downtown core—especially **Financial District North, Union Square, Financial District South, South of Market, and Civic Center**—while **industrial** parcels concentrate in **Bayview, Potrero Hill, South of Market, and Dogpatch/Central Waterfront**.
- The vacant-government footprint is concentrated in large special-purpose land areas such as **Treasure Island / Yerba Buena Island, Hunters Point, Bayview, Dogpatch, and Mission Bay**.

This is an important framing result: the assessor roll is not just a value dataset, but also a compact map of San Francisco's economic structure.


Source: https://thefrontsteps.com/san-francisco-real-estate-district-maps/  

![SF Real Estate District](https://i0.wp.com/thefrontsteps.com/wp-content/uploads/2021/12/San-Francisco-Real-Estate-Districts-Map.jpg?w=701ssl=1)

## 2. How do the various property types change over time?

This section moves from a single-year map to a historical view. Looking across roll years helps answer whether the city's property stock is stable or whether some categories expand, shrink, or respond to major shocks.

Two complementary views are useful:
- **absolute counts**, which show how many properties fall in each group;
- **shares of total properties**, which show the long-run composition of the city's property base.

In [ ]:
use_code_year = df.groupby(["closed_roll_year", "use_code"]).size().unstack(fill_value=0)
use_code_year.columns = use_code_year.columns.map(use_code_maps)
use_code_year = use_code_year[use_code_year.columns.sort_values()]
use_code_year.index = clean_year_index(use_code_year.index)
use_code_year.index.name = None 

display(
    style_table(
        use_code_year,
        caption="Property counts by type and roll year",
        formatters={col: fmt_int for col in use_code_year.columns},
    )
)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- LEFT: all property types ---
use_code_year.plot(ax=ax1, legend=False, linewidth=5)
ax1.set_title("Property Count by Type and Roll Year")
ax1.set_xlabel("Roll Year")
ax1.set_ylabel("Number of properties")
apply_standard_axis_style(ax1)

# --- RIGHT: non-residential categories only ---
subset = use_code_year.drop(["1 Single Family (SRES)", "2 Multi Family (MRES)"], axis=1)
subset.plot(ax=ax2, color=['C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8'], linewidth=5)
ax2.set_title("Non-Residential Property Count by Roll Year")
ax2.set_xlabel("Roll Year")
ax2.set_ylabel("Number of properties")
apply_standard_axis_style(ax2)

# --- LEGEND ---
handles, labels = ax2.get_legend_handles_labels()
extra_handles = [Line2D([0], [0], color="C0", linewidth=5), Line2D([0], [0], color="C1", linewidth=5)]
extra_labels = ["1 Single Family (SRES)", "2 Multi Family (MRES)"]

ax2.legend(
    extra_handles + handles,
    extra_labels + labels,
    title="Property type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
)

plt.tight_layout()
# plt.savefig("assets/02_property_counts_over_time.png")
plt.show()


In [ ]:
mean_props = use_code_year.div(use_code_year.sum(axis=1), axis=0).mean()

# Get top 2 indices (by value)
top2 = mean_props.nlargest(2).index

def autopct_func(pct):
    # This function doesn't know the label directly,
    # so we use a counter to track slices
    autopct_func.counter += 1
    label = mean_props.index[autopct_func.counter - 1]
    
    if label in top2:
        return f'{pct:.1f}%'
    else:
        return ''

autopct_func.counter = 0

fig, ax = plt.subplots(figsize=(7,5))

wedges, texts, autotexts = ax.pie(
    mean_props,
    autopct=autopct_func,
    labels=None
)

ax.set_title("Proportion of \nVarious Property Types")

ax.legend(
    wedges,
    mean_props.index,
    bbox_to_anchor=(0.95, 1),
    title='Property Type'
)

plt.tight_layout()
# plt.savefig("assets/03_average_property_type_shares.png")
plt.show()

#### Observations

The roll composition is remarkably stable, but the long-run trend still reveals a few meaningful structural shifts.

- In **2024**, the roll contains **154,886 single-family** records and **36,320 multi-family** records, so residential properties dominate the dataset by count.
- **Single-family** records fell from **140,514 in 2010** to **132,843 in 2011**, a drop of about **5.5%**, before recovering and reaching a new high by 2024.
- **Multi-family** records drifted slightly downward over time, from **37,941 in 2007** to **36,320 in 2024**, a decline of roughly **4.3%**.
- **Timeshare** properties decline materially after 2020, which may reflect pandemic-era stress on tourism-oriented property uses.
- The small specialist categories are far more cyclical. That matters because a change of a few hundred records can look large in percentage terms even when it is small relative to the whole tax base.

The main takeaway is that San Francisco's property inventory changes slowly. Most of the action in this notebook therefore comes not from dramatic shifts in counts, but from **where value is concentrated**, **how assessments differ across neighborhoods**, and **which parts of the market are actually transacting**.


## 3. Analysis: assessed values of various property types

After understanding **how many** properties exist in each category, the next question is **how much value** each category represents.

This section looks at:
- the **median assessed value** for an individual property in each category; and
- the **total assessed value** contributed by that category across the city.

These two views answer different questions:
- Median value helps describe the typical scale of a property type.
- Total value helps describe the category's contribution to the city's tax base.

**Data note:** the 2014 roll appears to have missing or unusable assessment values, so that year is excluded from the value trend analysis.

In [ ]:
# Median assessed value per property, shown in millions of USD for readability.
median_value_year = (
    df.loc[df["closed_roll_year"] != "2014-01-01"]
    .groupby(["closed_roll_year", "use_code"])["assessed_total"]
    .median()
    .unstack()
    / 1_000_000
)
median_value_year.columns = median_value_year.columns.map(use_code_maps)
median_value_year = median_value_year[median_value_year.columns.sort_values()].round(3)
median_value_year.index = clean_year_index(median_value_year.index)
median_value_year.index.name = None

display(
    style_table(
        median_value_year,
        caption="Median assessed value by property type and roll year",
        formatters={col: fmt_usd_m for col in median_value_year.columns},
    )
)


In [ ]:
# Total assessed value, shown in billions of USD for readability.
total_value_year = (
    df.loc[df["closed_roll_year"] != "2014-01-01"]
    .groupby(["closed_roll_year", "use_code"])["assessed_total"]
    .sum()
    .unstack()
    / 1_000_000_000
)
total_value_year.columns = total_value_year.columns.map(use_code_maps)
total_value_year = total_value_year[total_value_year.columns.sort_values()].round(3)
total_value_year.index = clean_year_index(total_value_year.index)
total_value_year.index.name = None

display(
    style_table(
        total_value_year,
        caption="Total assessed value by property type and roll year",
        formatters={col: fmt_usd_b for col in total_value_year.columns},
    )
)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

median_value_year.plot(ax=ax1, legend=False, linewidth=4)
ax1.set_title("Median Assessed Value by Property Type")
ax1.set_xlabel("Roll Year")
ax1.set_ylabel("Median assessed value")
apply_standard_axis_style(ax1, y_is_currency=True, currency_scale=1)
ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"${x:,.1f}M"))

total_value_year.plot(ax=ax2, legend=False, linewidth=4)
ax2.set_title("Total Assessed Value by Property Type")
ax2.set_xlabel("Roll Year")
ax2.set_ylabel("Total assessed value")
apply_standard_axis_style(ax2, y_is_currency=True, currency_scale=1)
ax2.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"${x:,.0f}B"))

ax2.legend(title="Property type", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

plt.tight_layout()
# plt.savefig("assets/04_median_and_total_assessed_value_by_type.png")
plt.show()


#### Observations

**At the individual-property level**

- The median assessed value of a **single-family** property rose from **$0.316M in 2007** to **$0.747M in 2024**, an increase of about **136%**.
- The median **multi-family** property rose from **$0.426M** to **$0.994M** over the same period, up roughly **133%**.
- Office and hotel parcels remain expensive on a per-record basis, reaching **$2.2M per Office parcel and $3.0M per Hotel parcel** in 2024. 
- Government-owned vacant parcels frequently have very low or zero assessed values, which is consistent with tax-exempt or administratively different treatment.

**At the aggregate citywide level**

- The total assessed value of **single-family housing** increased from **$58.1B in 2007** to **$154.0B in 2024**.
- **Multi-family housing** increased from **$25.8B** to **$74.7B**, while **office** starts from a very large **$18.4B** base in 2007 to **$59.7B** in 2024 despite having far fewer parcels.
- This distinction is crucial: a category can matter because it is **numerous** (single-family/multi-family), because it is **valuable per parcel** (office/hotel), or both.

This is why the notebook repeatedly separates **median parcel value** from **aggregate tax-base contribution**. The two metrics answer different policy questions.


## 4. Focus section: single-family homes and condominiums

The remainder of the notebook narrows the lens to two especially relevant housing segments:

- **Single-family homes** (`Dwelling`)
- **Condominiums** (`Condominium`)

This narrower focus is useful for downstream housing analysis because these two categories are more comparable to the kinds of properties households actually buy and sell. The notebook filters out records with implausibly small land, improvement, or total assessments so that vacant land, incomplete records, and likely non-comparable cases do not distort neighborhood comparisons.

In [ ]:
def percentile_clip(series: pd.Series, low: float = 0.1, high: float = 0.9) -> tuple[float, float]:
    """Return quantile clip bounds for a numeric series."""
    if isinstance(series, pd.DataFrame):
        series = series.iloc[:, 0]
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return (np.nan, np.nan)
    return float(s.quantile(low)), float(s.quantile(high))


def plot_metric_map(
    gdf: gpd.GeoDataFrame,
    metric_col: str,
    title: str,
    cmap: str = "RdYlBu_r",
    center_zero: bool = False,
    clip_quantiles: tuple[float, float] = (0.1, 0.9),
    marker_size: float = 10,
    legend_label: str | None = None,
    ax=None,
    vmin=None,
    vmax=None,
    show_legend: bool = True,
):
    """Plot a building-level metric on top of a lightweight basemap."""
    data = gdf.dropna(subset=[metric_col]).copy()
    if data.empty:
        raise ValueError(f"No valid rows available for {metric_col}.")

    data[metric_col] = pd.to_numeric(data[metric_col], errors="coerce")
    data = data.dropna(subset=[metric_col])
    if data.empty:
        raise ValueError(f"No numeric rows available for {metric_col}.")

    q_low, q_high = percentile_clip(data[metric_col], *clip_quantiles)
    clip_low = vmin if vmin is not None else q_low
    clip_high = vmax if vmax is not None else q_high

    if np.isnan(clip_low) or np.isnan(clip_high):
        raise ValueError(f"Could not compute clip bounds for {metric_col}.")
    if clip_low == clip_high:
        clip_low -= 1
        clip_high += 1

    data["metric_clipped"] = data[metric_col].clip(clip_low, clip_high)
    data_3857 = data.to_crs(epsg=3857)

    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 10))

    if center_zero:
        bound = max(abs(clip_low), abs(clip_high))
        norm = TwoSlopeNorm(vmin=-bound, vcenter=0.0, vmax=bound)
    else:
        norm = Normalize(vmin=clip_low, vmax=clip_high)

    data_3857.plot(
        ax=ax,
        column="metric_clipped",
        cmap=cmap,
        markersize=marker_size,
        alpha=0.9,
        legend=show_legend,
        norm=norm,
        legend_kwds={"shrink": 0.32, "label": legend_label or metric_col},
        zorder=3,
    )

    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, attribution=False)

    ax.set_title(title, fontsize=14)
    ax.set_axis_off()

    xmin, ymin, xmax, ymax = data_3857.total_bounds
    xpad = (xmax - xmin) * 0.05
    ypad = (ymax - ymin) * 0.05
    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim(ymin - ypad, ymax + ypad)

    return norm


def plot_metric_map_dual(
    gdf1: gpd.GeoDataFrame,
    gdf2: gpd.GeoDataFrame,
    title1: str = "San Francisco Single Family\nAssessed Property Value - Year 2024",
    title2: str = "San Francisco Condominium\nAssessed Property Value - Year 2024",
    metric_col: str = "assessed_total",
    cmap: str = "RdYlBu_r",
    bar_label: str = "Million USD",
    clip_quantiles: tuple[float, float] = (0.1, 0.9),
    marker_size: float = 2.2,
    output_png: str | None = None,
    dpi: int = 300,
):
    """Plot two maps side by side with a shared color scale."""
    combined_values = pd.concat(
        [pd.to_numeric(gdf1[metric_col], errors="coerce"),
         pd.to_numeric(gdf2[metric_col], errors="coerce")],
        ignore_index=True,
    ).dropna()

    if combined_values.empty:
        raise ValueError(f"No valid combined values available for {metric_col}.")

    shared_vmin, shared_vmax = percentile_clip(combined_values, *clip_quantiles)
    if shared_vmin == shared_vmax:
        shared_vmin -= 1
        shared_vmax += 1

    fig, axes = plt.subplots(1, 2, figsize=(21, 9))

    norm = plot_metric_map(
        gdf=gdf1,
        metric_col=metric_col,
        title=title1,
        cmap=cmap,
        center_zero=False,
        clip_quantiles=clip_quantiles,
        marker_size=marker_size,
        legend_label="",
        ax=axes[0],
        vmin=shared_vmin,
        vmax=shared_vmax,
        show_legend=False,
    )

    plot_metric_map(
        gdf=gdf2,
        metric_col=metric_col,
        title=title2,
        cmap=cmap,
        center_zero=False,
        clip_quantiles=clip_quantiles,
        marker_size=marker_size,
        legend_label="",
        ax=axes[1],
        vmin=shared_vmin,
        vmax=shared_vmax,
        show_legend=False,
    )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=axes,
        fraction=0.018,   # smaller color bar
        pad=0.012,
        aspect=25,        # taller / slimmer
    )
    cbar.set_label(bar_label, fontsize=14)
    cbar.ax.tick_params(labelsize=14)
    cbar.outline.set_linewidth(0.6)

    for ax in axes:
        ax.title.set_fontsize(14)

    fig.subplots_adjust(wspace=0.04, right=0.88)

    if output_png is not None:
        fig.savefig(output_png, dpi=dpi, bbox_inches="tight")

    plt.show()
    return fig, axes

Clean up the data for subsequent export

In [ ]:
# Focus on single-family residential records
df_SRES = df.loc[df['use_code'] == 'SRES'].copy()

# Exclude records with implausibly small assessment components (less than $100).
# This helps remove likely vacant/partial records that are not suitable for housing comparison.
df_SRES = df_SRES[df_SRES['assessed_total'] >= 100]
df_SRES = df_SRES[df_SRES['assessed_improvement_value'] >= 100]
df_SRES = df_SRES[df_SRES['assessed_land_value'] >= 100]

# exclude buildings that were 'welfare', 'collge/university', 'religious' etc.
# keeping only 'NA' (unspecified) or '11' (owner occupied)
df_SRES = df_SRES[(df_SRES['exemption_code'].isna()) | (df_SRES['exemption_code']=='11')]

# Building age should be measured relative to the roll year being analyzed, not the current calendar year.
df_SRES['age'] = df_SRES['closed_roll_year'].dt.year - df_SRES['year_property_built']

# Guard against division by zero or missing building area when constructing assessed value per square foot.
valid_area = df_SRES['property_area'].where(df_SRES['property_area'] > 0)
df_SRES['psf'] = df_SRES['assessed_total'] / valid_area


Extract Year 2024 Only

In [ ]:
# Extract Year 2024 data only
df_2024_SRES = df_SRES[df_SRES['closed_roll_year'] == '2024-01-01'].copy()

## add a bit more information
df_2024_SRES['assessor_neighborhood'] = df_2024_SRES['assessor_neighborhood_code'] + ' ' + df_2024_SRES['assessor_neighborhood'] 
df_2024_SRES['assessed_total'] = df_2024_SRES['assessed_total']/10**6

# Keep only the two owner-occupier-oriented classes used in the housing comparison section.
gdf_2024_dwell = gpd.GeoDataFrame(
    df_2024_SRES.loc[df_2024_SRES['property_class_code_definition'] == 'Dwelling'],
    geometry='geometry',
    crs='EPSG:4326'
)
gdf_2024_condo = gpd.GeoDataFrame(
    df_2024_SRES.loc[df_2024_SRES['property_class_code_definition'] == 'Condominium'],
    geometry='geometry',
    crs='EPSG:4326'
)

In [ ]:
# Approximate a recent-sales subset.
# The current cutoff keeps properties sold since 2021, which is close to a trailing 5-year window
# relative to when this notebook was prepared in 2026.
recent_sale_cutoff = pd.Timestamp('2021-01-01')

gdf_2024_condo_5y = gdf_2024_condo[gdf_2024_condo['current_sales_date'] > recent_sale_cutoff].copy()
gdf_2024_dwell_5y = gdf_2024_dwell[gdf_2024_dwell['current_sales_date'] > recent_sale_cutoff].copy()

### 4(A) Where are the more expensive and the more affordable neighborhoods?

We will first examine the assessed property values across San Francisco. This is done using the most recent data (Year 2024) and is a snapshot for a point in time.

In [ ]:
plot_metric_map_dual(
    gdf1 = gdf_2024_dwell,
    gdf2 = gdf_2024_condo,
    title1 = "Single Family\nAssessed Property Value - Year 2024",
    title2 = "Condominium\nAssessed Property Value - Year 2024",
    metric_col = "assessed_total",
    cmap = "RdYlBu_r",
    bar_label = "Million USD",
    clip_quantiles = (0.1, 0.9),
    # output_png = 'assets/05_total_assessed_value_maps_single_family_vs_condo.png'
)

In [ ]:
# Neighborhood summary table used for ranking and interpretation
dwell_neighborhood = gdf_2024_dwell.groupby("assessor_neighborhood").agg(
    median_value=("assessed_total", "median"),
    num_properties=("assessed_total", "size")).sort_values("median_value", ascending=False)
dwell_neighborhood = dwell_neighborhood[dwell_neighborhood.num_properties > 100]


In [ ]:
display(
    style_table(
        dwell_neighborhood.head(10).rename(columns={
            "median_value": "Median assessed value (USD millions)",
            "num_properties": "Property count",
        }),
        caption="Single-family neighborhoods with the highest 2024 median assessed value",
        formatters={
            "Median assessed value (USD millions)": fmt_usd_m,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
display(
    style_table(
        dwell_neighborhood.tail(10)[::-1].rename(columns={
            "median_value": "Median assessed value (USD millions)",
            "num_properties": "Property count",
        }),
        caption="Single-family neighborhoods with the lowest 2024 median assessed value",
        formatters={
            "Median assessed value (USD millions)": fmt_usd_m,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
# Neighborhood summary table used for ranking and interpretation
condo_neighborhood = gdf_2024_condo.groupby('assessor_neighborhood').agg(
    median_value=('assessed_total', 'median'),
    num_properties=('assessed_total', 'size')).sort_values('median_value', ascending=False)
condo_neighborhood = condo_neighborhood[condo_neighborhood.num_properties>100]

In [ ]:
display(
    style_table(
        condo_neighborhood.head(10).rename(columns={
            "median_value": "Median assessed value (USD millions)",
            "num_properties": "Property count",
        }),
        caption="Condominium neighborhoods with the highest 2024 median assessed value",
        formatters={
            "Median assessed value (USD millions)": fmt_usd_m,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
display(
    style_table(
        condo_neighborhood.tail(10)[::-1].rename(columns={
            "median_value": "Median assessed value (USD millions)",
            "num_properties": "Property count",
        }),
        caption="Condominium neighborhoods with the lowest 2024 median assessed value",
        formatters={
            "Median assessed value (USD millions)": fmt_usd_m,
            "Property count": fmt_int,
        },
    )
)


#### Observations

- For **single-family homes**, **Presidio Heights** leads at **$4.729M**, while **Hunters Point** is at **$0.342M**. That is a gap of roughly **13.8x**.
- For **condominiums**, the spread is narrower: **Presidio Heights** leads at **$1.605M**, versus **$0.449M** in **Mount Davidson Manor**, or about **3.6x**.
- That narrower condo spread is an early hint that condo pricing is driven more by **location and building typology**, whereas single-family values combine both **location** and **lot / structure scale**.

### 4(B) What about assessed price per square foot?

Total assessed value is influenced by home size. To normalize across differently sized homes, this section uses **assessed value per square foot**.

This is **not** the same thing as observed market sale price per square foot, but it is still analytically useful because it helps separate two stories:
1. neighborhoods where homes are valuable mainly because they are **large**, and  
2. neighborhoods where each unit of housing space is itself **highly valued**.


In [ ]:
plot_metric_map_dual(
    gdf1 = gdf_2024_dwell,
    gdf2 = gdf_2024_condo,
    title1 = "Single Family\nAssessed Price per Square Foot - Year 2024",
    title2 = "Condominium\nAssessed Price per Square Foot - Year 2024",
    metric_col = "psf",
    cmap = "RdYlBu_r",
    bar_label = "USD",
    clip_quantiles = (0.1, 0.9),
    # output_png = 'assets/06_assessed_value_per_sqft_maps.png'
)

In [ ]:
# Neighborhood summary table used for ranking and interpretation
dwell_neighborhood = gdf_2024_dwell.groupby('assessor_neighborhood').agg(
    median_value=('psf', 'median'),
    num_properties=('psf', 'size')).sort_values('median_value', ascending=False)
dwell_neighborhood = dwell_neighborhood[dwell_neighborhood.num_properties>100]

In [ ]:
display(
    style_table(
        dwell_neighborhood.head(10).rename(columns={
            "median_value": "Median assessed value per sq ft",
            "num_properties": "Property count",
        }),
        caption="Single-family neighborhoods with the highest 2024 assessed value per square foot",
        formatters={
            "Median assessed value per sq ft": fmt_psf,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
display(
    style_table(
        dwell_neighborhood.tail(10)[::-1].rename(columns={
            "median_value": "Median assessed value per sq ft",
            "num_properties": "Property count",
        }),
        caption="Single-family neighborhoods with the lowest 2024 assessed value per square foot",
        formatters={
            "Median assessed value per sq ft": fmt_psf,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
# Neighborhood summary table used for ranking and interpretation
condo_neighborhood = gdf_2024_condo.groupby('assessor_neighborhood').agg(
    median_value=('psf', 'median'),
    num_properties=('psf', 'size')).sort_values('median_value', ascending=False)
condo_neighborhood = condo_neighborhood[condo_neighborhood.num_properties>100]

In [ ]:
display(
    style_table(
        condo_neighborhood.head(10).rename(columns={
            "median_value": "Median assessed value per sq ft",
            "num_properties": "Property count",
        }),
        caption="Condominium neighborhoods with the highest 2024 assessed value per square foot",
        formatters={
            "Median assessed value per sq ft": fmt_psf,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
display(
    style_table(
        condo_neighborhood.tail(10)[::-1].rename(columns={
            "median_value": "Median assessed value per sq ft",
            "num_properties": "Property count",
        }),
        caption="Condominium neighborhoods with the lowest 2024 assessed value per square foot",
        formatters={
            "Median assessed value per sq ft": fmt_psf,
            "Property count": fmt_int,
        },
    )
)



#### Observations

- For single-family homes, the most expensive space is in **Telegraph Hill ($1,199/sq ft)**, **Pacific Heights ($1,167/sq ft)**, and **Presidio Heights ($1,134/sq ft)**.
- The low end is concentrated in the southeast: **Hunters Point ($253/sq ft)**, **Bayview ($305/sq ft)**, and **Visitacion Valley ($322/sq ft)**.
- For condos, the top tier is much more downtown-oriented: **Financial District South ($1,145/sq ft)**, **Central Waterfront / Dogpatch ($1,145/sq ft)**, **South Beach ($1,142/sq ft)**, and **Mission Bay ($1,047/sq ft)**.
- That pattern says the condo market is pricing access to jobs, waterfront redevelopment, and dense amenity clusters much more directly than the single-family market.

### 4(C) How old are the buildings?

Building age adds an important structural dimension. It can proxy for neighborhood development history, redevelopment pressure, housing form, preservation status, and in some cases prestige or functional obsolescence.

Looking at the age profile helps answer three different questions:

- Which neighborhoods are dominated by **legacy housing stock**?
- Which parts of the city show evidence of **recent development waves**?
- How different is the development history of **single-family districts** versus **condominium-heavy districts**?


In [ ]:
plot_metric_map_dual(
    gdf1 = gdf_2024_dwell,
    gdf2 = gdf_2024_condo,
    title1 = "Single Family\nAge of Building - Year 2024",
    title2 = "Condominium\nAge of Building - Year 2024",
    metric_col = "age",
    cmap = "viridis_r",
    bar_label = "Year",
    clip_quantiles = (0.1, 0.9),
    # output_png='assets/07_building_age_maps.png'
)

In [ ]:
# Neighborhood summary table used for ranking and interpretation
dwell_neighborhood = gdf_2024_dwell.groupby('assessor_neighborhood').agg(
    median_age_of_building=('age', 'median'),
    num_properties=('age', 'size')).sort_values('median_age_of_building', ascending=False)
dwell_neighborhood = dwell_neighborhood[dwell_neighborhood.num_properties>100]

In [ ]:
display(
    style_table(
        dwell_neighborhood.head(10).rename(columns={
            "median_age_of_building": "Median building age",
            "num_properties": "Property count",
        }),
        caption="Oldest single-family neighborhoods in the 2024 roll",
        formatters={
            "Median building age": fmt_years,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
display(
    style_table(
        dwell_neighborhood.tail(10)[::-1].rename(columns={
            "median_age_of_building": "Median building age",
            "num_properties": "Property count",
        }),
        caption="Youngest single-family neighborhoods in the 2024 roll",
        formatters={
            "Median building age": fmt_years,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
# Neighborhood summary table used for ranking and interpretation
condo_neighborhood = gdf_2024_condo.groupby('assessor_neighborhood').agg(
    median_age_of_building=('age', 'median'),
    num_properties=('age', 'size')).sort_values('median_age_of_building', ascending=False)
condo_neighborhood = condo_neighborhood[condo_neighborhood.num_properties>100]

In [ ]:
display(
    style_table(
        condo_neighborhood.head(10).rename(columns={
            "median_age_of_building": "Median building age",
            "num_properties": "Property count",
        }),
        caption="Oldest condominium neighborhoods in the 2024 roll",
        formatters={
            "Median building age": fmt_years,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
display(
    style_table(
        condo_neighborhood.tail(10)[::-1].rename(columns={
            "median_age_of_building": "Median building age",
            "num_properties": "Property count",
        }),
        caption="Youngest condominium neighborhoods in the 2024 roll",
        formatters={
            "Median building age": fmt_years,
            "Property count": fmt_int,
        },
    )
)


#### Observations

- The oldest single-family neighborhoods in the table—**Lower Pacific Heights, Hayes Valley, North Panhandle, Inner Mission, and Haight Ashbury**—cluster around a median building age of **124 years**.
- The youngest single-family neighborhood in the filtered table is **Hunters Point** at **36 years**, followed by **Diamond Heights (59 years)** and **Forest Knolls (63 years)**.
- The condo stock is dramatically newer in redevelopment districts: **Hunters Point (8 years)**, **Stonestown (9 years)**, **Central Waterfront/Dogpatch (12 years)**, **Financial District South (15 years)**, and **Mission Bay (15 years)**.
- That contrast tells us the condo market is much more intertwined with **post-2000 redevelopment**, while single-family housing remains predominantly an **older, slower-changing stock**.

### 4(D) Where are the recently sold homes?

Assessed values are most informative about current market conditions when a property has transacted relatively recently. Under California's Proposition 13 system, long-held properties can carry assessed values that remain anchored to an older base-year value and then rise only gradually unless ownership changes or new construction occurs.

This section therefore isolates homes and condos sold within the last five years, then examines their 2024 assessed values. The objective is not to reconstruct exact market sale prices, but to focus on the portion of the roll whose assessments are most likely to be closer to contemporary market conditions.




In [ ]:
plot_metric_map_dual(
    gdf1 = gdf_2024_dwell_5y,
    gdf2 = gdf_2024_condo_5y,
    title1 = "Single Family\nBought & Sold within Last 5 Years\n Assessed Property Value - Year 2024",
    title2 = "Condominium\nBought & Sold within Last 5 Years\n Assessed Property Value - Year 2024",
    metric_col = "assessed_total",
    cmap = "RdYlBu_r",
    bar_label = "Million USD",
    clip_quantiles = (0.1, 0.9),
    # output_png='assets/08_recent_sales_maps_last_5_years.png'
)

In [ ]:
# Neighborhood summary table used for ranking and interpretation
dwell_neighborhood = gdf_2024_dwell_5y.groupby('assessor_neighborhood').agg(
    median_value=('assessed_total', 'median'),
    num_properties=('assessed_total', 'size')).sort_values('num_properties', ascending=False)


In [ ]:
display(
    style_table(
        dwell_neighborhood.head(10).rename(columns={
            "median_value": "Median assessed value (USD millions)",
            "num_properties": "Recent transaction count",
        }),
        caption="Single-family neighborhoods with the highest number of recent transactions",
        formatters={
            "Median assessed value (USD millions)": fmt_usd_m,
            "Recent transaction count": fmt_int,
        },
    )
)


In [ ]:
# Neighborhood summary table used for ranking and interpretation
condo_neighborhood = gdf_2024_condo_5y.groupby('assessor_neighborhood').agg(
    median_value=('assessed_total', 'median'),
    num_properties=('assessed_total', 'size')).sort_values('num_properties', ascending=False)

In [ ]:
display(
    style_table(
        condo_neighborhood.head(10).rename(columns={
            "median_value": "Median assessed value (USD millions)",
            "num_properties": "Recent transaction count",
        }),
        caption="Condominium neighborhoods with the highest number of recent transactions",
        formatters={
            "Median assessed value (USD millions)": fmt_usd_m,
            "Recent transaction count": fmt_int,
        },
    )
)


In [ ]:
# Neighborhood summary table used for ranking and interpretation
dwell_neighborhood_all = gdf_2024_dwell.groupby("assessor_neighborhood").agg(
    median_value_all=("assessed_total", "median"))
# Neighborhood summary table used for ranking and interpretation
dwell_neighborhood_5y = gdf_2024_dwell_5y.groupby('assessor_neighborhood').agg(
    median_value_5y=("assessed_total", "median"))

a = dwell_neighborhood_5y.merge(dwell_neighborhood_all, 'left', on='assessor_neighborhood')

# Neighborhood summary table used for ranking and interpretation
condo_neighborhood_all = gdf_2024_condo.groupby("assessor_neighborhood").agg(
    median_value_all=("assessed_total", "median"))
# Neighborhood summary table used for ranking and interpretation
condo_neighborhood_5y = gdf_2024_condo_5y.groupby('assessor_neighborhood').agg(
    median_value_5y=("assessed_total", "median"))

b = condo_neighborhood_5y.merge(condo_neighborhood_all, 'left', on='assessor_neighborhood')

fig, (ax1, ax2) = plt.subplots(1,2, figsize=(10, 5))

ax1.scatter(a.median_value_all, a.median_value_5y)

# Get limits
xmin = min(a.median_value_all.min(), a.median_value_5y.min())
xmax = max(a.median_value_all.max(), a.median_value_5y.max())

# Plot diagonal line (gradient = 1)
ax1.plot([xmin, xmax], [xmin, xmax], c='red', linestyle='--')

ax1.set_xlabel("Median value based on all available Single-Family\nMillion USD")
ax1.set_ylabel("Median value based on \nrecently transacted Single-Family only\nMillion USD")
ax1.set_title("Effect of Proposition 13 on\nAssessed Value for Neighborhoods\n(Single Family)")


ax2.scatter(b.median_value_all, b.median_value_5y, c='C1')

# Get limits
xmin = min(b.median_value_all.min(), b.median_value_5y.min())
xmax = max(b.median_value_all.max(), b.median_value_5y.max())

# Plot diagonal line (gradient = 1)
ax2.plot([xmin, xmax], [xmin, xmax], c='red', linestyle='--')

ax2.set_xlabel("Median value based on all available Condo\nMillion USD")
ax2.set_ylabel("Median value based on \nrecently transacted Condo only\nMillion USD")
ax2.set_title("Effect of Proposition 13 on\nAssessed Value for Neighborhoods\n(Condo)")

plt.tight_layout()
# plt.savefig('assets/09_prop13_effect_recent_vs_all_stock.png')
plt.show()

#### Observations

This section is most useful as a **market-facing correction** to the all-stock value maps shown earlier. In California's Proposition 13 system, assessed values are strongly shaped by transaction history. Long-held homes can remain tied to an older base-year assessment, while recently sold homes are re-assessed much closer to contemporary market conditions.

- For **single-family homes**, the neighborhoods with the largest number of recent transactions include **Bernal Heights**, **Noe Valley**, and **Excelsior**. Their median assessed values for recently sold homes fall roughly in the **$1.1M-$2.7M** range, which is more aligned with lived market expectations than the all-stock neighborhood medians.
- For **condominiums**, the most active recent-sale neighborhoods include **South Beach**, **Pacific Heights**, and **Inner Mission**. Even within this more actively traded segment, the typical recently transacted unit is still assessed at around **$1M or more**, underscoring how expensive the city remains even in the more liquid condo market.
- The scatter comparison against all-stock neighborhood medians shows the Proposition 13 effect directly: neighborhoods with many long-held properties can look artificially inexpensive on the roll because legacy assessments are still anchored to older purchase dates.
- That means the assessor roll is very good for comparing **relative structure and geography**, but it must be interpreted carefully for **absolute price level** discussions. A neighborhood can look cheaper on the roll not because it is truly cheap today, but because much of its stock has not reset to current market value.
- Filtering to properties sold in the last five years therefore gives a view that is closer to the **current marginal market** — the part of the stock that is actually resetting values and absorbing new demand.
- The trade-off is that this filter sharply reduces sample size. It improves market relevance, but it also means the analysis is now describing a thinner, more selective subset of the housing stock.

That final point matters for the next section. Once the notebook distinguishes **total stock** from **recently circulating stock**, it becomes possible to ask a more structural question: **which housing segments actually turn over, and who appears to be buying them?**


### 4(E) Property Turn-over Rates and Owner-Occupancy Rates

Section 4(D) showed that recently transacted homes can look very different from the full housing stock. This section takes that insight one step further by comparing three linked dimensions:

1. the **composition of the housing stock**,  
2. the **recent turnover rate**, and  
3. the **share of properties that appear owner-occupied** using the homeowner exemption flag.

Together, these measures help separate the market into two analytically distinct layers:

- a **large legacy stock** that exists physically but turns over slowly, and  
- a **circulating stock** that re-enters the market more often and does most of the work of absorbing new demand.

The owner-occupancy measure should be treated as a **proxy**, not a perfect census of tenure. Even so, it is informative when read alongside turnover, because it helps distinguish:

- segments dominated by **stable owner households**,  
- segments with a larger **renter or investor footprint**, and  
- segments where homes appear to function more as **active market inventory** than as long-held neighborhood stock.


In [ ]:
summary1 = pd.DataFrame({
    "Units in analysis": [
        gdf_2024_dwell.shape[0],
        gdf_2024_condo.shape[0],
    ],
    "Owner-occupied units": [
        (gdf_2024_dwell.exemption_code == "11").sum(),
        (gdf_2024_condo.exemption_code == "11").sum(),
    ],
    "Owner-occupied share": [
        (gdf_2024_dwell.exemption_code == "11").sum() / gdf_2024_dwell.shape[0] * 100,
        (gdf_2024_condo.exemption_code == "11").sum() / gdf_2024_condo.shape[0] * 100,
    ],
    "Units sold in last 5 years": [
        gdf_2024_dwell_5y.shape[0],
        gdf_2024_condo_5y.shape[0],
    ],
    "5-year turnover rate": [
        gdf_2024_dwell_5y.shape[0] / gdf_2024_dwell.shape[0] * 100,
        gdf_2024_condo_5y.shape[0] / gdf_2024_condo.shape[0] * 100,
    ],
    "Units NOT sold in last 5 years": [
        gdf_2024_dwell.shape[0]-gdf_2024_dwell_5y.shape[0],
        gdf_2024_condo.shape[0]-gdf_2024_condo_5y.shape[0],
    ],
}, index=["Single-family", "Condominium"])

display(
    style_table(
        summary1.iloc[:, :-1],
        caption="Occupancy and recent turnover by housing type",
        formatters={
            "Units in analysis": fmt_int,
            "Owner-occupied units": fmt_int,
            "Owner-occupied share": fmt_pct,
            "Units sold in last 5 years": fmt_int,
            "5-year turnover rate": fmt_pct,
        },
    )
)


In [ ]:
summary2 = pd.DataFrame({
    "Units sold in last 5 years": [
        gdf_2024_dwell_5y.shape[0],
        gdf_2024_condo_5y.shape[0],
    ],
    "Recent purchases that are owner-occupied": [
        (gdf_2024_dwell_5y.exemption_code == "11").sum(),
        (gdf_2024_condo_5y.exemption_code == "11").sum(),
    ],
    "Owner-occupied share": [
        (gdf_2024_dwell_5y.exemption_code == "11").sum() / gdf_2024_dwell_5y.shape[0] * 100,
        (gdf_2024_condo_5y.exemption_code == "11").sum() / gdf_2024_condo_5y.shape[0] * 100,
    ],
    "Recent purchases that appear renter-occupied": [
        gdf_2024_dwell_5y.shape[0] - (gdf_2024_dwell_5y.exemption_code == "11").sum(),
        gdf_2024_condo_5y.shape[0] - (gdf_2024_condo_5y.exemption_code == "11").sum(),
    ],
    "Renter-occupied share": [
        (gdf_2024_dwell_5y.shape[0] - (gdf_2024_dwell_5y.exemption_code == "11").sum()) / gdf_2024_dwell_5y.shape[0] * 100,
        (gdf_2024_condo_5y.shape[0] - (gdf_2024_condo_5y.exemption_code == "11").sum()) / gdf_2024_condo_5y.shape[0] * 100,
    ],
}, index=["Single-family", "Condominium"])

display(
    style_table(
        summary2,
        caption="Occupancy mix among units purchased in the last 5 years",
        formatters={
            "Units sold in last 5 years": fmt_int,
            "Recent purchases that are owner-occupied": fmt_int,
            "Owner-occupied share": fmt_pct,
            "Recent purchases that appear renter-occupied": fmt_int,
            "Renter-occupied share": fmt_pct,
        },
    )
)


In [ ]:
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

values = pd.concat([
    summary1['Units NOT sold in last 5 years'],
    summary1['Units sold in last 5 years'][::-1]
])

colors = [
    (*mcolors.to_rgba('C0')[:3], 0.5),  # SF Not Sold
    (*mcolors.to_rgba('C1')[:3], 0.5),  # Condo Not Sold
    (*mcolors.to_rgba('C1')[:3], 1),  # Condo Sold
    (*mcolors.to_rgba('C0')[:3], 1),  # SF Sold
]

labels = [
    'Single-Family Units',
    'Condo Units',
    'Condo Sold Last 5 Years',
    'Single-Family Sold Last 5 Years'
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# --- Pie chart ---
wedges, texts = ax1.pie(
    values,
    colors=colors,
    startangle=90
)

ax1.set_title("Market Composition and\nTurnover by Property Type")

ax1.legend(
    wedges,
    labels,
    title="Legend",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    ncol=2,
    frameon=False,
    fontsize=10,
    title_fontsize=11
)

# --- Stacked bar chart ---
tmp = summary2.iloc[:, [1, 3]]
plt.bar(tmp.index, tmp.iloc[:,0], color=[(*mcolors.to_rgba('C0')[:3], 1),  (*mcolors.to_rgba('C1')[:3], 1),], hatch='/')
plt.bar(tmp.index, tmp.iloc[:,1], bottom=tmp.iloc[:,0], color=[(*mcolors.to_rgba('C0')[:3], 1),  (*mcolors.to_rgba('C1')[:3], 1),])

ax2.set_title("Occupancy Profile of New Buyers")
ax2.set_ylabel("Count")
ax2.set_xlabel("")
ax2.tick_params(axis='x', rotation=45)

legend_elements = [
    Patch(facecolor='white', edgecolor='black', label='Investor-Owned'),
    Patch(facecolor='white', edgecolor='black', hatch='/', label='Owner-Occupied')
]

ax2.legend(
    handles=legend_elements,
    title="Buyer Type",
    bbox_to_anchor=(1, 0.5),
    loc='center left'
)

plt.tight_layout()
# plt.savefig('assets/10_market_composition_and_buyer_profile.png')
plt.show()

#### Observations

This section is one of the clearest in the notebook because it shows that San Francisco is not operating as one unified housing market. Instead, the results support a **Two Housing Circuits** interpretation: a **legacy circuit** of tightly held homes, and a **circulation circuit** of homes that return to market and absorb new demand.

- The analysis set contains **94,844 single-family homes** and **52,042 condominiums**. In other words, **the ratio of single-family to condominium is about 2:1**.
- The apparent owner-occupied share is **55.6%** for single-family homes, versus **34.2%** for condos — a **21.4 percentage-point gap**. That does not mean every condo is investor-held, but it does show that the condo stock is much less rooted in owner-occupation than the single-family stock.
- Recent turnover differs even more sharply: **8,531 single-family homes** sold in the last five years (**9.0%** of the stock), while **10,520 condos** sold (**20.2%** of the stock). The condo segment therefore turns over at more than **2.2 times** the single-family rate.
- This is the key structural result: **single-family homes dominate the stock, but condos dominate the flow**. San Francisco has far more single-family homes in total, yet over a five-year window the smaller condo stock generates more transactions.
- Among recently purchased units, only **27.8%** of single-family acquisitions and **24.0%** of condo acquisitions appear owner-occupied based on exemption code `11`. The remaining shares — **72.2%** and **76.0%** respectively — appear not owner-occupied. In other words, **for every 4 residential properties transacted in San Francisco, 3 are bought by investors and 1 are bought by owner-occupier.** The situation is the same for both the single-family market and the condo market.

##### What the numbers imply

These three measures — stock size, turnover, and buyer profile — fit together naturally.

- **Single-family housing looks like legacy housing.** It is the larger stock, it has the higher owner-occupancy share, and it turns over slowly. Much of it appears to sit outside the city's active for-sale market in any given multi-year period.
- **Condominiums look like circulation housing.** The stock is smaller, but it recycles through the market much more quickly and appears more exposed to renter and investor demand.
- The practical implication is that **accessible market supply is much smaller than physical stock**. Headline housing totals overstate how much inventory is actually available to a household trying to buy into the city.

##### Why this leads directly to Section 4(F)

Section 4(E) identifies the pattern, but it does not yet show the underlying time structure. If single-family homes really belong more to a legacy circuit, then their holding periods should be much longer and less market-like. If condos really belong more to a circulation circuit, then their holding periods should fall away in a more regular churn pattern.

That is exactly what Section 4(F) tests next by looking at **years since last sale**.

##### A particularly important finding: investors appear active in both segments

One of the strongest results here is that recently transacted units in **both** property types look overwhelmingly non-owner-occupied after purchase.

That matters because it cuts against a simplified narrative in which single-family homes are mainly contested by owner-occupiers while condos are mainly contested by investors. The evidence suggests something more nuanced: once a home enters the **circulation circuit**, both single-family homes and condos appear to face substantial competition from buyers who do not immediately occupy the property themselves.

A household trying to buy into San Francisco therefore appears to face that competition across the broader active market, not only in the condo segment.


### 4(F) No. of Years Since Last Sale

Section 4(E) showed that the single-family stock is much larger but turns over far less, while the condo stock is smaller but circulates much more actively. This section examines that same contrast more directly through **holding periods**.

There are two complementary views below:

- a **2024 spatial view** showing where long-held and recently traded properties are located, and  
- a **distributional view using 2019 data** to look more cleanly at the shape of holding periods over time.

The 2019 distributions are especially useful for the decay analysis because they allow the older holding-period tail to be examined without leaning too heavily on the newest year alone. The core question is simple:

**Do single-family homes and condos thin out through time in the same way once they enter the stock, or do they follow different ownership rhythms?**


In [ ]:
gdf_2024_dwell['years_since_last_sale'] = (pd.to_datetime('2024-12-31') - gdf_2024_dwell['current_sales_date']).dt.days / 365.25
gdf_2024_condo['years_since_last_sale'] = (pd.to_datetime('2024-12-31') - gdf_2024_condo['current_sales_date']).dt.days / 365.25

In [ ]:
plot_metric_map_dual(
    gdf1 = gdf_2024_dwell,
    gdf2 = gdf_2024_condo,
    title1 = "Single Family\nNo. of Years Since Last Sale (Year 2024 Data)",
    title2 = "Condominium\nNo. of Years Since Last Sale (Year 2024 Data)",
    metric_col = "years_since_last_sale",
    cmap = "viridis_r",
    bar_label = "Year",
    clip_quantiles = (0.1, 0.9),
    # output_png='assets/11_years_since_last_sale_maps.png'
)

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# dwell = gdf_2024_dwell['years_since_last_sale'].dropna()
# condo = gdf_2024_condo['years_since_last_sale'].dropna()

# bins = np.linspace(0, 40, 41)

# fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

# # --- Dwelling ---
# axes[0].hist(dwell, bins=bins)
# axes[0].axvline(5, linestyle='--', c="red", linewidth=2)  # <-- FIX
# axes[0].set_title('Single-Family Dwellings\nYears Since Last Sale (Y2024 Data)', fontsize=12)
# axes[0].set_xlabel('Years')
# axes[0].set_ylabel('Count')
# axes[0].grid(alpha=0.3)

# # --- Condo ---
# axes[1].hist(condo, bins=bins)
# axes[1].axvline(5, linestyle='--', c="red", linewidth=2)  # <-- ADD HERE TOO
# axes[1].set_title('Condominiums\nYears Since Last Sale (Y2024 Data)', fontsize=12)
# axes[1].set_xlabel('Years')
# axes[1].set_ylabel('Count')
# axes[1].grid(alpha=0.3)

# fig.suptitle('Distribution of Holding Periods (Years Since Last Sale) (Y2024 Data)', fontsize=14)

# plt.tight_layout()
# plt.show()

In [ ]:
# Extract Year 2019 data only
df_2019_SRES = df_SRES[df_SRES['closed_roll_year'] == '2019-01-01'].copy()

# Keep only the two owner-occupier-oriented classes used in the housing comparison section.
gdf_2019_dwell = gpd.GeoDataFrame(
    df_2019_SRES.loc[df_2019_SRES['property_class_code_definition'] == 'Dwelling'],
    geometry='geometry',
    crs='EPSG:4326'
)
gdf_2019_condo = gpd.GeoDataFrame(
    df_2019_SRES.loc[df_2019_SRES['property_class_code_definition'] == 'Condominium'],
    geometry='geometry',
    crs='EPSG:4326'
)

In [ ]:
gdf_2019_dwell['years_since_last_sale'] = (pd.to_datetime('2019-12-31') - gdf_2019_dwell['current_sales_date']).dt.days / 365.25
gdf_2019_condo['years_since_last_sale'] = (pd.to_datetime('2019-12-31') - gdf_2019_condo['current_sales_date']).dt.days / 365.25

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# --- Data ---
dwell = gdf_2019_dwell['years_since_last_sale'].dropna()
condo = gdf_2019_condo['years_since_last_sale'].dropna()

bins = np.linspace(0, 35, 36)

# --- Decay function ---
def exp_decay(x, A, lam):
    return A * np.exp(-lam * x)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, data, title, color in zip(
    axes,
    [dwell, condo],
    ['Single-Family Dwellings', 'Condominiums'],
    ['C0', 'C1']
):
    # Histogram
    counts, edges, _ = ax.hist(data, bins=bins, alpha=1, color=color)

    # Prepare for fitting
    x = (edges[:-1] + edges[1:]) / 2
    y = counts

    mask = y > 0
    x_fit = x[mask]
    y_fit = y[mask]

    # Fit decay
    params, _ = curve_fit(exp_decay, x_fit, y_fit, p0=(y.max(), 0.1))
    A, lam = params

    # Smooth curve
    x_smooth = np.linspace(0, 35, 200)
    y_smooth = exp_decay(x_smooth, A, lam)

    ax.plot(x_smooth, y_smooth, linewidth=2, label=f'λ = {lam:.3f}', color="red")

    # Styling
    ax.set_title(f'{title}\nYears Since Last Sale', fontsize=12)
    ax.set_xlabel('Years')
    ax.grid(alpha=0.3)
    ax.legend()

# Shared y-label
axes[0].set_ylabel('Count')
axes[1].set_ylabel('Count')

# Overall title
fig.suptitle('Distribution of Holding Periods (Years Since Last Sale) (Y2019 Data)', fontsize=14)

plt.tight_layout()
# plt.savefig('assets/12_holding_period_histograms_2019.png')
plt.show()

In [ ]:
### pseudo count
# Parameters
A = 2369
lam = 0.078
n_buckets = 33
# Buckets: 0 to 32
x = np.arange(0, n_buckets)
# Exponential decay
y = A * np.exp(-lam * (x))   # start at A for bucket 1
# Optional: round to integers
y_int = np.round(y).astype(int)
# Create pandas Series
dwell_pseudo = np.repeat(x, y_int)

In [ ]:
dwell = gdf_2019_dwell['years_since_last_sale'].dropna()
condo = gdf_2019_condo['years_since_last_sale'].dropna()

bins = np.linspace(0, 35, 36)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

# --- Dwelling ---
axes[0].hist(dwell, bins=bins, color='C0', alpha=0.6, label='Legacy (Never for Sale)')
axes[0].hist(dwell_pseudo, bins=bins,  color='C0', alpha=1, label='Circulating (Estimate)')

axes[0].set_title('Single-Family Dwellings\nYears Since Last Sale', fontsize=12)
axes[0].set_xlabel('Years')
axes[0].set_ylabel('Count')
axes[0].grid(alpha=0.3)
axes[0].legend()

# --- Condo ---
axes[1].hist(condo, bins=bins, color='C1')
axes[1].set_title('Condominiums\nYears Since Last Sale', fontsize=12)
axes[1].set_xlabel('Years')
axes[1].set_ylabel('Count')
axes[1].grid(alpha=0.3)

fig.suptitle('Distribution of Holding Periods (Years Since Last Sale) (Y2019 Data)', fontsize=14)

plt.tight_layout()
# plt.savefig('assets/13_decay_function_fit.png')
plt.show()

#### Observations

Section 4(E) established the **Two Housing Circuits** idea using stock composition, turnover, and buyer profile. This companion section deepens that interpretation by asking a more dynamic question: **how long do homes stay in the same ownership before returning to the market?**

The key insight is that the two housing types do not merely differ in who occupies them today. They differ in the **rhythm of ownership turnover itself**.

#### Main observations from the holding-period figures

- The **condo distribution** is much more consistent with a **circulation market**: many units have sold recently, and the number of units falls off steadily as holding periods lengthen. The histogram is broadly consistent with a time-dependent exponential decay pattern.
- The **single-family distribution** is not. Instead of showing a clean churn profile, it suggests the presence of a large stock of parcels that remain in the same ownership for very long periods.
- That is exactly the pattern hinted at in Section 4(E): the single-family segment contains many homes that exist in the stock but do not participate regularly in market circulation.

#### Decay-function interpretation

The decay-function analysis is useful because it captures not just how many homes sold recently, but how quickly ownership cohorts thin out as time passes.

- For **condominiums**, the fitted exponential decay parameter is about **λ = 0.078**.
- Interpreted intuitively, that means the count of condos in a given "not sold since" cohort shrinks by about **7.8% per additional year** in exponential terms.
- This corresponds to an implied **half-life of roughly 8.9 years**.

That is strong evidence of a genuine **circulation circuit**. Condos behave like a housing type that continually recycles through the market. This helps explain why they play such a large role in absorbing new demand, setting marginal prices, and determining who can still enter the city as an owner.

By contrast:

- For **single-family homes**, the fitted parameter is only about **λ = 0.011**.
- That implies a notional half-life of roughly **63.0 years**, though the fit is weak and should be interpreted cautiously.

The weak fit is itself informative. Single-family homes do **not** appear to follow a clean exponential-decay process because a substantial part of the stock is not behaving like normal circulating inventory at all. Instead, it behaves like **legacy housing**: homes held for decades by the same owners, whether as long-term residences or as long-term rental assets.

#### Why the decay result matters

This section confirms and sharpens the interpretation from Section 4(E):

- the condo market is not just **less owner-occupied**; it is also **structurally more recyclable**;
- the single-family market is not just **more owner-occupied**; it is also **structurally less liquid**, with a much larger component of long-duration holding.

In that sense, the housing constraint is not simply the total number of homes. It is the size of the **circulation circuit**. In a market where prices are set at the margin, what matters most is not only how much housing exists, but how much housing is actually **circulating**.

The decay curves suggest that San Francisco's circulating stock is concentrated disproportionately in the condo segment, while the single-family segment contains a much larger legacy stock that participates only weakly in present-day turnover.

A large share of the single-family stock therefore appears effectively **not for sale** in ordinary market conditions. Some of these homes are long-held owner residences. Others are likely long-held rental assets. In both cases, they count toward the city's housing inventory, but they contribute relatively little to current market access.

That helps explain how San Francisco can have a large residential base and still feel exceptionally difficult to enter as a buyer: the relevant competition is occurring in a much thinner market than the headline stock totals imply.

##### Broader civic interpretation

These results also point to a deeper social balance that cities must manage:

- the **legacy circuit** preserves continuity, neighborhood memory, and long-term rootedness;  
- the **circulation circuit** enables entry, adaptation, and demographic renewal.

A healthy city needs both. But if too much of the stock remains locked in the legacy circuit, the city becomes harder for newcomers to join. And if too much of the circulation circuit is captured by investor-oriented demand, the city risks becoming accessible mainly as a place to rent or invest, rather than as a place where new households can put down long-term ownership roots.


## Final synthesis of the seven major conclusions

### 1. San Francisco’s housing geography is sharply unequal

The city contains large neighborhood gaps even within the same property type. Single-family medians range from **$4.729M in Presidio Heights** to **$0.342M in Hunters Point**. On a per-square-foot basis, single-family values range from about **$1,199/sq ft in Telegraph Hill** to **$253/sq ft in Hunters Point**. These are distinct submarkets, not small deviations around a citywide average.

### 2. Single-family homes and condos play different structural roles

Across the report, the same contrast appears repeatedly.

- **Age:** single-family neighborhoods are often much older; many condo-heavy districts are products of more recent redevelopment.
- **Turnover:** condos show a **20.2%** five-year turnover rate versus **9.0%** for single-family homes.
- **Occupancy:** apparent owner occupancy is **55.6%** for single-family homes and **34.2%** for condos.
- **Recent purchases:** only **27.8%** of recently purchased single-family homes and **24.0%** of recently purchased condos appear owner-occupied.
- **Holding periods:** condos show a much clearer circulation pattern with **λ = 0.078** and an implied half-life of **8.9 years**, while single-family homes show a much flatter pattern with **λ = 0.011** and a notional half-life of **63.0 years**.

These results support the report’s core interpretation: **single-family homes behave more like the legacy circuit, while condos behave more like the circulation circuit**.

### 3. The active market is much smaller than the physical stock

This is the most important practical result in the report.

San Francisco contains about **94,844** single-family homes, yet only **8,531** sold in the last five years. By contrast, the much smaller condo stock of **52,042** units generated **10,520** sales over the same period. The homes that actually absorb demand and help set current prices come from a much thinner pool than headline stock counts suggest.

### 4. Proposition 13 is not a background detail; it shapes the reading of the entire dataset

The recent-sales comparison shows that all-stock neighborhood medians can materially understate current conditions in neighborhoods with many long-held properties. That does not make the dataset less useful. It means the roll should be read as both a valuation record and a record of ownership history.

### 5. Investor-oriented outcomes appear in both property types

A striking result is that recently purchased units in **both** segments appear overwhelmingly outside the owner-occupied category. The implication is that investor or non-owner-occupying demand is not confined to condos. Once a home enters the active market, both property types appear exposed to similar competitive pressure.

### 6. The main bottleneck is circulation, not just stock

The housing problem described here is not only about how many units exist. It is also about how many units actually return to market. The condo segment does far more of that work. The single-family segment contains a much larger share of long-held stock that contributes relatively little to current market access.

### 7. The Two Housing Circuits framework ties the report together

The value maps, age patterns, recent-sales filter, turnover tables, buyer-profile proxy, and holding-period distributions all point in the same direction.

- The **legacy circuit** is older, more tightly held, and slower to recycle.
- The **circulation circuit** turns over more often, carries more of the burden of entry, and is where competition is most concentrated.

That is the main conclusion of the project. San Francisco’s housing challenge is not only that housing is expensive. It is that a large share of the stock is structurally slow-moving, while the smaller circulating share bears most of the pressure of entry, investment demand, and price discovery.